# DBSCAN. Density-Based Spatial Clustering

---

## Overview

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) discovers clusters of arbitrary shape and identifies outliers automatically.

**Key parameters:**
- $\varepsilon$ (`eps`): neighborhood radius
- `min_samples`: minimum points to form a dense region

**Point types:**
- **Core point**: has $\geq$ `min_samples` neighbors within $\varepsilon$
- **Border point**: within $\varepsilon$ of a core point but not a core point itself
- **Noise point**: not reachable from any core point. labeled $-1$

**Algorithm:**
1. For each unvisited core point, start a new cluster
2. Expand via BFS through density-connected points
3. Points unreachable from any core point are labeled noise

---

**Dataset:** Food Nutrition (`FOOD-DATA-GROUP1.csv`) or synthetic blobs with noise  
**Task:** Find density-based food clusters and identify outliers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.unsupervised_learning import DBSCAN, PCA
from rice_ml.preprocess import StandardScaler

In [ ]:
try:
    df = pd.read_csv('../../../data/FOOD-DATA-GROUP1.csv')
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    X = df[num_cols].dropna().values.astype(float)
    print(f'Loaded food nutrition dataset (FOOD-DATA-GROUP1): {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import make_blobs
    rng = np.random.default_rng(5)
    X_blobs, _ = make_blobs(n_samples=150, centers=3, n_features=2, random_state=5)
    # Add noise points
    noise = rng.uniform(-10, 10, (20, 2))
    X = np.vstack([X_blobs, noise])
    print('CSV not found. using synthetic blobs with noise')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_2d = PCA(n_components=2).fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.5, color='steelblue')
plt.xlabel('PC 1', fontsize=15)
plt.ylabel('PC 2', fontsize=15)
plt.title('Data: PCA Projection (Before Clustering)', fontsize=18)
plt.show()

## Fit DBSCAN

In [ ]:
db = DBSCAN(eps=0.5, min_samples=5)
labels = db.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))

print(f'Clusters found: {n_clusters}')
print(f'Noise points:   {n_noise} ({100*n_noise/len(labels):.1f}%)')

In [ ]:
colors = ['red', 'lightseagreen', 'steelblue', 'magenta', 'orange',
          'purple', 'brown', 'pink', 'cyan', 'lime']

plt.figure(figsize=(10, 8))
unique_labels = set(labels)
for lbl in unique_labels:
    mask = labels == lbl
    if lbl == -1:
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c='black', marker='x', s=50, label='Noise', alpha=0.6)
    else:
        plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=colors[lbl % len(colors)], label=f'Cluster {lbl}', alpha=0.7)

plt.xlabel('PC 1', fontsize=15)
plt.ylabel('PC 2', fontsize=15)
plt.title(f'DBSCAN: {n_clusters} Clusters + {n_noise} Noise Points', fontsize=18)
plt.legend(fontsize=12)
plt.show()

## Effect of `eps` on Clustering

Smaller `eps` → more noise; larger `eps` → fewer, larger clusters.

In [ ]:
eps_values = [0.2, 0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, len(eps_values), figsize=(18, 5))

for ax, eps in zip(axes, eps_values):
    lbs = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
    n_c = len(set(lbs)) - (1 if -1 in lbs else 0)
    n_n = int(np.sum(lbs == -1))
    for lbl in set(lbs):
        mask = lbs == lbl
        color = 'black' if lbl == -1 else colors[lbl % len(colors)]
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, alpha=0.6, s=20)
    ax.set_title(f'eps={eps}\n{n_c} clusters, {n_n} noise', fontsize=12)

plt.suptitle('DBSCAN: Effect of eps', fontsize=16)
plt.tight_layout()
plt.show()

## Interpretation

- DBSCAN finds clusters of **arbitrary shape**. unlike $k$-Means which assumes spheres.
- Noise points (labeled $-1$) are outliers not belonging to any dense region. useful for anomaly detection.
- `eps` and `min_samples` must be tuned together: use a **$k$-distance plot** to choose `eps`.